In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from utils.plots import calculate_accuracy, plot_confusion_matrices, plot_distributions, plot_accuracy, plot_metric_heatmap, plot_precision_recall_curves, plot_cooccurrence, plot_error_distribution, GT_COLS, PRED_COLS, LABELS

In [97]:
def create_row_annotation(config):
    annotations = []

    if bool(config.get("use_instructions", 0)):
        annotations.append("with instructions")
    else:
        annotations.append("without instructions")

    n_demos = config.get("number_of_demonstrations")
    annotations.append(f"number of demonstrations: {n_demos}")
    if n_demos > 0:
        t = ""
        demo_type = config.get("type_of_demonstrations")
        match demo_type:
            case -1:
                t = "negative"
            case 0:
                t = "mixed"
            case 1:
                t = "positive"

        annotations.append(f"type of demonstrations: {t}")


    return "\n".join(annotations)

def collect_jobs(base_dir):
    jobs = {}

    base_path = Path(base_dir)

    for model_family in base_path.iterdir():
        if not model_family.is_dir():
            continue

        for model in model_family.iterdir():
            if not model.is_dir():
                continue

            for job in model.iterdir():
                if not job.is_dir():
                    continue

                results_file = job / "result.csv"
                config_file = job / "config.json"

                if results_file.exists() and config_file.exists():

                    if model not in jobs.keys():
                        jobs[str(model)] = []
                        
                    job_id = job.name

                    jobs[model].append({
                        "job_id": job_id,
                        "model_family": model_family.name,
                        "model": model.name,
                        "path": str(job),
                        "result_csv": str(results_file),
                        "config_json": str(config_file),
                    })

    return jobs

In [101]:
finished_jobs = collect_jobs("./outputs").get("Qwen2.5-7B-Instruct")
finished_jobs = finished_jobs[:2]
num_jobs = len(finished_jobs)

fig, axes = plt.subplots(num_jobs, 3, figsize=(16, 5 * num_jobs))
if num_jobs == 1:
    axes = [axes]

print(finished_jobs)

fig.suptitle(finished_jobs[0]["model"], fontsize=16)

for i, job_info in enumerate(finished_jobs):
    with open(job_info["config_json"], "r", encoding="utf-8") as config_json:
        config = json.loads("\n".join(config_json.readlines()))
        
    df = pd.read_csv(job_info["result_csv"], sep=";")
    plot_confusion_matrices(df, axes[i], use_title=True if i == 0 else False, xlabel="Predicted" if i == (len(csvs) - 1) else "", ylabel="True")

    axes[i, 0].annotate(
    create_row_annotation(config),
    xy=(-0.3, 0.5),
    xycoords='axes fraction',
    fontsize=12,
    ha='right',
    va='center',
    rotation=90
)
    
fig.tight_layout()
plt.show()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
def build_figure_for_csv(csvs,
                         labels=LABELS,
                         cols1=GT_COLS,
                         cols2=PRED_COLS,
                         conf=True, acc=True, dist=True):

    filepath, model = csvs
    
    df = pd.read_csv(filepath, sep=";")

    # Layout:
    # Row 0: confusion matrices (3 plots)
    # Row 1: accuracy (1 plot spanning all columns)
    # Row 2-4: distributions (3 rows x 2 columns)

    num_rows = 0
    if conf:
        num_rows += 1
    if acc:
        num_rows += 1
    if dist:
        num_rows += 3

    fig = plt.figure(figsize=(16, num_rows * 6))
    gs = fig.add_gridspec(num_rows, 3)

    # --- Confusion matrices ---
    if conf:
        cm_axes = [
            fig.add_subplot(gs[0, i]) for i in range(3)
        ]
        plot_confusion_matrices(df, cm_axes, labels, cols1, cols2)

    # --- Accuracy (span full width) ---
    if acc:
        acc_ax = fig.add_subplot(gs[1 if conf else 0, :])
        plot_accuracy(df, acc_ax, cols1, cols2)

    # --- Distributions ---
    if dist:
        dist_axes = [
            [fig.add_subplot(gs[num_rows - 3 + i, 0]),
             fig.add_subplot(gs[num_rows - 3 + i, 1])]
            for i in range(len(cols1))
        ]
    
        plot_distributions(df,
                           pd.DataFrame(dist_axes).values,
                           labels,
                           cols1, cols2)

    fig.suptitle(model, fontsize=16)
    fig.tight_layout()

    return fig


def run_on_csvs(csv_paths, conf=True, acc=True, dist=True):
    figs = []

    for path in csv_paths:
        fig = build_figure_for_csv(path, conf=conf, acc=acc, dist=dist)
        figs.append(fig)

    return figs

#run_on_csvs(csvs, acc=False, dist=False)